# Recession Model: Random Forest

In [ ]:
#Standard libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pandas import read_csv
import time
sns.set(style='whitegrid', palette='muted')
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 10, 6

from sklearn import metrics
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import SGDClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from xgboost import XGBClassifier
from sklearn.decomposition import PCA


In [ ]:
# Imports data
macro = pd.read_csv('US_FRED_data_1970_2020.csv', header=0, index_col=0, parse_dates=True)
macro.shape

In [ ]:
macro

In [ ]:
# Imports data
tran_code = pd.read_csv('US_FRED_data_transform_code.csv', header=0, index_col=0, parse_dates=False)
tran_code.shape

In [ ]:
tran_code

In [ ]:
def data_transform(x, tran_code):
    
    if tran_code == 1:
        x_tr = x        
        
    elif tran_code == 2:
        x_tr = x.diff()
            
    elif tran_code == 3:
        x_tr = x.diff().diff()
        
    elif tran_code == 4:
        x_tr = np.log(x)
        
    elif tran_code == 5:
        x_tr = np.log(x).diff()*100
    
    elif tran_code == 6:
        x_tr = np.log(x).diff().diff()*100
    
    else:
        x_tr = (x.pct_change()-1)*100
    
    return x_tr    

In [ ]:
macro_tr=[]

for col in macro.columns:
    tr_code = tran_code[col].values
    data_tr = data_transform(macro[col], tr_code)
    macro_tr.append(data_tr)

macro_tr = pd.DataFrame(macro_tr).T

In [ ]:
macro_tr

In [ ]:
# Imports data
gdp = pd.read_csv('US_recession_1970_2020.csv', header=0, index_col=0, parse_dates=True)
gdp.shape

In [ ]:
gdp

In [ ]:
macro_final = pd.concat([macro_tr, gdp], axis=1)
macro_final

In [ ]:
# Show rows and columns
print("Rows, Columns:");print(macro_final.shape);print("\n")

# Describe DataFrame columns
print("Columns:");print(macro_final.columns);print("\n")

# Show info on DataFrame
print("Info:");print(macro_final.info(max_cols=1000)); print("\n")

# Count Non-NA values
print("Non-NA:");print(macro_final.count()); print("\n")

# Show head
print("Head");print(macro_final.head()); print("\n")

# Show tail
print("Tail");print(macro_final.tail());print("\n")

# Show summary statistics
print("Summary statistics:");print(macro_final.describe());print("\n")

In [ ]:
#Define Data for Machine Learning

X_start = 2  # 1970-M3
X_lag = 1

# Features and Target
X, y = macro_final.iloc[X_start:,:-1].shift(X_lag).dropna(), macro_final.iloc[X_start+X_lag:,125]


In [ ]:
X

In [ ]:
y

# Principal Component Analysis (PCA)

In [ ]:
steps_X = [('scaler', StandardScaler(copy=True, with_mean=True, with_std=True)),
           ('pca', PCA(n_components = 1, random_state=1))]

pl_pca_X = Pipeline(steps_X)

X_pca_model = pl_pca_X.fit(X)

print('n_components:', 1, '  ', 'variance explained:', '%.3f' %X_pca_model.steps[1][1].explained_variance_ratio_.sum())

X_pca = X_pca_model.transform(X)

In [ ]:
for n_comp in [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,30,50,125]:

    steps_X = [('scaler', StandardScaler(copy=True, with_mean=True, with_std=True)),
               ('pca', PCA(n_components = n_comp, random_state=1))]
    
    pl_pca_X = Pipeline(steps_X)
    X_pca_model = pl_pca_X.fit(X)
    
    print('n_components:', n_comp, '  ', 'variance explained:', '%.3f' %X_pca_model.steps[1][1].explained_variance_ratio_.sum())


# Define Training, Validation, and Test Sets

In [ ]:
train_pct = 0.5
valid_pct = 0.25
test_pct = 0.25

n_train = int(train_pct*len(X))
n_valid = int(valid_pct*len(X))
n_test = len(X)-n_train-n_valid

print(n_train, n_valid, n_test)

In [ ]:
#n_trains = n_train
#n_records = len(X)

#j = 0
#for i in range(n_trains, n_records):    
#    #train, test = X[0:i], X[i:i+1]   #expanding window
#    train, test = X[j:i], X[i:i+1]   #rolling window
#    print('train=%d, test=%d' % (len(train), len(test)))
#    j += 1

# Grid Search for Optimal Hyperparameters

In [ ]:
# pipeline
def pipeline(config):

    # unpack config
    n_trees, n_comps = config

    # Steps
    steps = [('scaler', StandardScaler(copy=True, with_mean=True, with_std=True)),
             ('pca', PCA(n_components=n_comps, random_state=1)),
             ('rfc', RandomForestClassifier(n_estimators=n_trees, random_state=1))
            ]

    pipeline = Pipeline(steps)

    return pipeline

In [ ]:
def walk_forward_validation(cfg):
    
    n_trains = n_train

    n_records = n_train + n_valid

    y_test_list = []

    y_pred_list = []

    y_prob_list = []

    j = 0

    for i in range(n_trains, n_records):
    
        X_train, X_test, y_train, y_test = X[j:i], X[i:i+1], y[j:i], y[i:i+1]
       
        model = pipeline(cfg).fit(X_train, y_train)        
    
        y_pred = model.predict(X_test)
   
        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred_list.extend(y_pred)
    
        y_prob_list.extend(y_prob)
    
        y_test_list.extend(y_test)
    
        j += 1

    score_acc = metrics.accuracy_score(y_test_list, y_pred_list)
    
    print(' > %.3f' % score_acc)
    
    return score_acc 

In [ ]:
# score a model, return None on failure
def repeat_evaluate(config, n_repeats=1):
    # convert config to a key
    key = str(config)
    # fit and evaluate the model n times
    scores = [walk_forward_validation(config) for _ in range(n_repeats)]
    # summarize score
    result = np.mean(scores)
    print('> Model[%s] %.3f' % (key, result))
    return (key, result)

In [ ]:
# grid search configs
def grid_search(cfg_list):
    # evaluate configs
    scores = [repeat_evaluate(cfg) for cfg in cfg_list]
    # sort configs by score_acc, asc
    scores.sort(key=lambda tup: tup[1])
    return scores

In [ ]:
# create a list of configs to try
def model_configs():
    # define scope of configs
    
    n_trees = [100]

#    n_trees = [100, 200, 300]

    n_comps = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
        
    # create configs
    configs = list()
    for i in n_trees:
        for j in n_comps:
            cfg = [i, j]
            configs.append(cfg)
    print('Total configs: %d' % len(configs))
    return configs

In [ ]:
# model configs
cfg_list = model_configs()

In [ ]:
cfg_list = model_configs()
# grid search
scores = grid_search(cfg_list)
print('done')

In [ ]:
scores

# Final Model Estimation & Evaluation

In [ ]:
# Pipeline

# Steps

steps_final = [('scaler', StandardScaler(copy=True, with_mean=True, with_std=True)),
               ('pca', PCA(n_components=8, random_state=1)),
               ('rfc', RandomForestClassifier(n_estimators=100, random_state=1))
              ]

pipeline_final = Pipeline(steps_final)

In [ ]:
start=time.time()    
   
n_trains = n_train + n_valid

n_records = n_train + n_valid + n_test

y_test_list = []
    
y_pred_list = []
    
y_prob_list = []

j = 0

for i in range(n_trains, n_records):
    
    X_train, X_test, y_train, y_test = X[j:i], X[i:i+1], y[j:i], y[i:i+1]
       
    model = pipeline_final.fit(X_train, y_train)        
    
    y_pred = model.predict(X_test)
     
    y_prob = model.predict_proba(X_test)[:, 1]
    
    y_pred_list.extend(y_pred)
        
    y_prob_list.extend(y_prob)
    
    y_test_list.extend(y_test)

    j += 1
    
end=time.time()

print("Running Time:", end - start)

In [ ]:
# create confustion matrix

CM = metrics.confusion_matrix(y_test_list, y_pred_list)
TN = CM[0][0]
FN = CM[1][0]
TP = CM[1][1]
FP = CM[0][1]

fig, ax=plt.subplots()
sns.heatmap(pd.DataFrame(metrics.confusion_matrix(y_test_list, y_pred_list)), annot=True, cmap="Blues" ,fmt='g')
plt.title('Confusion matrix'); plt.ylabel('Actual label'); plt.xlabel('Predicted label')
ax.xaxis.set_ticklabels(['Expansion', 'Recession']); ax.yaxis.set_ticklabels(['Expansion', 'Recession'])

Accuracy = metrics.accuracy_score(y_test_list, y_pred_list)*100
Recession_Precision = metrics.precision_score(y_test_list, y_pred_list)*100
Recession_Recall = metrics.recall_score(y_test_list, y_pred_list)*100
Recession_F1 = metrics.f1_score(y_test_list, y_pred_list)*100
Expansion_Precision = TN/(TN+FN)*100
Expansion_Recall = TN/(TN+FP)*100
Expansion_F1 = 2*(TN/(TN+FN))*(TN/(TN+FP))/((TN/(TN+FN)) +(TN/(TN+FP)))*100
ROC_AUC =  metrics.roc_auc_score(y_test_list, y_prob_list)
                                 
print("Overall Accuracy:", '%.3f' %Accuracy)                      
print("Recession: Precision:   ", '%.3f' % Recession_Precision)
print("Recession: Recall:      ", '%.3f' % Recession_Recall)
print("Recession: F1 score:    ", '%.3f' % Recession_F1)
print("Expansion: Precision: ", '%.3f' % Expansion_Precision)
print("Expansion: Recall:    ", '%.3f' % Expansion_Recall)
print("Expansion: F1 score:  ", '%.3f' % Expansion_F1)
print("ROC_AUC:         ", '%.3f' % ROC_AUC)

In [ ]:
fpr, tpr, _=metrics.roc_curve(y_test_list, y_prob_list)
auc=metrics.roc_auc_score(y_test_list, y_prob_list)
plt.plot(fpr,tpr,label="AUC="+str(auc))
plt.legend(loc=4)
plt.plot([0, 1], [0, 1], linestyle='--') # plot no skill
plt.title('ROC-Curve')
plt.show()

In [ ]:
result = pd.DataFrame({'Recession_Actual': y[-n_test:], 'Recession_Predicted': y_pred_list, 'Recession_Probability': y_prob_list}, columns=['Recession_Actual', 'Recession_Predicted', 'Recession_Probability'])
#result.to_csv('recession_result_rfc.csv')
pd.options.display.max_rows=300
result

In [ ]:
result['Recession_Actual'].plot(legend=True)
result['Recession_Probability'].plot(legend=True)

In [ ]:
result.corr()